## Notes

Want economic analysis function perhaps?

If price is low - buy electricity to charge battery
If price is low and PV output is high --> charge battery
If price is high and PV output is high --> straight to grid and dispatch battery


In [1]:
import pandas as pd
import numpy as np
import cvxpy as cp

In [80]:
price_data = pd.read_csv("Sylmar RTM LMP Data.csv")
prices = price_data['VALUE']

In [81]:
solar_generation = pd.read_csv('Symlar_pvwatts_hourly.csv')
# "Month","Day","Hour",
# "Beam Irradiance (W/m2)","Diffuse Irradiance (W/m2)",
# "Ambient Temperature (C)","Wind Speed (m/s)","Albedo","Plane of Array Irradiance (W/m2)",
# "Cell Temperature (C)",
# "DC Array Output (W)","AC System Output (W)"

solar_output = pd.DataFrame()
solar_output["Year 1"] = solar_generation["AC System Output (W)"]

In [82]:
# Calculate the solar output over the 30 year period for the degredation
lifespan = 30
panel_degredation = 0.005 # /yr
om_escalation = 0.01 # % /yr


### CASE 1

In [83]:
# Battery Specifications

Bat_capex =  99.865 # $/kW
Bat_OM =  3994.6 # $/kW-yr
Bat_LCOE = 4144.345  # $/MWh

SGIP = 0.85 # $/Wh
ITC = 0.06 # % of installation cost

bat_cap = 1000 # kW

bat_capex =  Bat_capex * bat_cap * (1 - ITC)



In [84]:
P_max = 1.0 # MW
E_max = 4.0 # MWh
eta_c = 0.95
eta_d = 0.95
SOC_0 = 0.5 * E_max
soc_min = 0.1 * E_max
dt = 1/12
H = 288


In [85]:
import cvxpy as cp
import numpy as np

T = 12
x = cp.Variable(T, nonneg=True)
price = cp.Parameter(T, value=np.ones(T))

prob = cp.Problem(
    cp.Maximize(cp.sum(price * x)),
    [x <= 1]
)

prob.solve(solver=cp.OSQP)
print("Solved OK")


Solved OK


/Users/badynmilstein-touesnard/.julia/conda/3/aarch64/lib/python3.12/site-packages/cvxpy/expressions/expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 2 times so far.

  warnings.warn(msg, UserWarning)


In [ ]:
# Maximizing daily revenue

H = 288                 # 1 day horizon (5-min data)
dt = 5 / 60
soc_init = SOC_0        # initial SOC (MWh)
price_base = prices.mean()   # OK to compute once
P_base = P_max
E_base = E_max


results = []

# ROLLING HORIZON LOOP
for start in range(0, len(prices) - H, H):

    prices_h = prices.iloc[start:start + H].to_numpy()
    price_scaled = prices_h / price_base   # numpy array, length H

    charge = cp.Variable(H)
    discharge = cp.Variable(H)
    soc = cp.Variable(H)

    # Constraints
    constraints = [
        soc[0] == soc_init / E_base,

        soc[1:] <= soc[:-1]
            + (charge[:-1] * eta_c * dt) / E_base
            - (discharge[:-1] / eta_d * dt) / E_base,

        soc[1:] >= soc[:-1]
            + (charge[:-1] * eta_c * dt) / E_base
            - (discharge[:-1] / eta_d * dt) / E_base,

        soc >= 0,
        soc <= 1,

        charge >= 0,
        discharge >= 0,
        charge <= 1,
        discharge <= 1,
    ]

    # Objective
    objective = cp.Maximize(
        cp.sum(
            cp.multiply(prices_h, (discharge * P_base * eta_d - charge * P_base / eta_c)) * dt
        )
    ) 


    # Solve
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP)


    results.append({
        "charge": charge.value,
        "discharge": discharge.value,
        "soc": soc.value,
        "start_index": start,
    })

    soc_init = soc.value[-1] * E_base


/Users/badynmilstein-touesnard/.julia/conda/3/aarch64/lib/python3.12/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


In [87]:
print(type(prices.index))
print(prices.index[:5])


<class 'pandas.core.indexes.range.RangeIndex'>
RangeIndex(start=0, stop=5, step=1)


In [88]:
start_time = "2023-01-01 00:00"  # choose your dataset start
freq = "5T"  # 5-minute intervals

prices.index = pd.date_range(start=start_time, periods=len(prices), freq=freq)

/var/folders/w9/vwctfdqs0v11qkc_klj078540000gn/T/ipykernel_97230/1597950459.py:4: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  prices.index = pd.date_range(start=start_time, periods=len(prices), freq=freq)


In [89]:
time_index = prices.index[:len(prices) - H]

charge_all = np.concatenate([r["charge"] for r in results])
discharge_all = np.concatenate([r["discharge"] for r in results])
soc_all = np.concatenate([r["soc"] for r in results])

dispatch = pd.DataFrame(
    {
        "price": prices.iloc[:len(charge_all)],
        "charge_MW": charge_all * P_base,
        "discharge_MW": discharge_all * P_base,
        "SOC_MWh": soc_all * E_base,
    },
    index=prices.index[:len(charge_all)]
)

dispatch["charge_MWh"] = dispatch["charge_MW"] * dt
dispatch["discharge_MWh"] = dispatch["discharge_MW"] * dt


In [90]:
dispatch["charge_cost_$"] = (
    dispatch["charge_MWh"] / eta_c
) * dispatch["price"]

dispatch["discharge_revenue_$"] = (
    dispatch["discharge_MWh"] * eta_d
) * dispatch["price"]

dispatch["net_revenue_$"] = (
    dispatch["discharge_revenue_$"]
    - dispatch["charge_cost_$"]
)


In [91]:
cycle_cost_per_MWh = 5  # $/MWh for degredation 

dispatch["degradation_$"] = (
    dispatch["charge_MWh"] + dispatch["discharge_MWh"]
) * cycle_cost_per_MWh

dispatch["net_revenue_after_deg_$"] = (
    dispatch["net_revenue_$"] - dispatch["degradation_$"]
)



In [92]:
daily = dispatch.resample("D").sum()
annual = dispatch.resample("Y").sum()
annual_revenue = annual["net_revenue_after_deg_$"].sum()


/var/folders/w9/vwctfdqs0v11qkc_klj078540000gn/T/ipykernel_97230/4249702191.py:2: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual = dispatch.resample("Y").sum()


In [93]:
print(annual)

                   price     charge_MW  discharge_MW        SOC_MWh  \
2023-12-31  4.599494e+06  31390.797934  28720.630254  162718.541847   

             charge_MWh  discharge_MWh  charge_cost_$  discharge_revenue_$  \
2023-12-31  2615.899828    2393.385854   66966.412128        122739.232748   

            net_revenue_$  degradation_$  net_revenue_after_deg_$  
2023-12-31   55772.820619   25046.428412             30726.392208  


In [94]:
case_1 = pd.DataFrame(
    index=range(1, lifespan + 1)
)

case_1["Yearly Bat OM"] = (
    Bat_OM
    * bat_cap
    * (1 + om_escalation) ** (case_1.index - 1)
)

print(case_1)

    Yearly Bat OM
1    3.994600e+06
2    4.034546e+06
3    4.074891e+06
4    4.115640e+06
5    4.156797e+06
6    4.198365e+06
7    4.240348e+06
8    4.282752e+06
9    4.325579e+06
10   4.368835e+06
11   4.412524e+06
12   4.456649e+06
13   4.501215e+06
14   4.546227e+06
15   4.591690e+06
16   4.637607e+06
17   4.683983e+06
18   4.730822e+06
19   4.778131e+06
20   4.825912e+06
21   4.874171e+06
22   4.922913e+06
23   4.972142e+06
24   5.021863e+06
25   5.072082e+06
26   5.122803e+06
27   5.174031e+06
28   5.225771e+06
29   5.278029e+06
30   5.330809e+06


In [95]:
IRR = 0.12
print(bat_capex)
npv = -bat_capex

for t in range(1, 31):
    revenue_t = annual["net_revenue_after_deg_$"].sum()
    om_t = case_1["Yearly Bat OM"][t]
    npv += (revenue_t - om_t) / (1 + IRR)**t

print("NPV: $", npv)

93873.09999999999
NPV: $ -34527178.57048208


Want to look at 
• Installation Costs (ITC is here)
• Annual operating cost differences
• NPV impact of augmentation timing

### CASE 2

In [70]:
# SOLAR PARAMETERS

solar_installed_capacity = 4000  # kW
solar_capacity_factor = 0.202
duration = 4  # hours
pv_dc = solar_installed_capacity * 1000
inverter = 3000  # kW
dc_ac_ratio = solar_installed_capacity/inverter

Co_LCOE = 79.872 # $/MWh
Co_CAPEX = 2275.408 * 0.5 # $/kW
Co_FIXED_OM = 60.325 # $/kW-year

ac_inverter = pv_dc / dc_ac_ratio

co_cap = Co_CAPEX * solar_installed_capacity * (1 - ITC)  


In [71]:
years = np.arange(lifespan)
panel_degredation = 0.005 # /yr

degradation_factors = (1 - panel_degredation) ** years

yearly_output = (
    solar_output["Year 1"].sum()
    * degradation_factors
)

case_2 = pd.DataFrame(
    {"Output AC (kW)": yearly_output},
    index=range(1, lifespan + 1)
)

case_2["Yearly OM"] = (
    Co_FIXED_OM
    * solar_installed_capacity
    * (1 + om_escalation) ** (case_2.index - 1)
)

case_2["Revenue"] = (
    Co_LCOE
    * case_2["Output AC (kW)"]
)

print(case_2)

    Output AC (kW)      Yearly OM       Revenue
1     7.071413e+09  241300.000000  5.648079e+11
2     7.036055e+09  243713.000000  5.619838e+11
3     7.000875e+09  246150.130000  5.591739e+11
4     6.965871e+09  248611.631300  5.563780e+11
5     6.931041e+09  251097.747613  5.535961e+11
6     6.896386e+09  253608.725089  5.508282e+11
7     6.861904e+09  256144.812340  5.480740e+11
8     6.827595e+09  258706.260463  5.453337e+11
9     6.793457e+09  261293.323068  5.426070e+11
10    6.759490e+09  263906.256299  5.398940e+11
11    6.725692e+09  266545.318862  5.371945e+11
12    6.692064e+09  269210.772050  5.345085e+11
13    6.658603e+09  271902.879771  5.318360e+11
14    6.625310e+09  274621.908569  5.291768e+11
15    6.592184e+09  277368.127654  5.265309e+11
16    6.559223e+09  280141.808931  5.238982e+11
17    6.526427e+09  282943.227020  5.212788e+11
18    6.493795e+09  285772.659290  5.186724e+11
19    6.461326e+09  288630.385883  5.160790e+11
20    6.429019e+09  291516.689742  5.134

In [75]:

P_max = 1.0 # MW
E_max = 4.0 # MWh
eta_c = 0.95
eta_d = 0.95
soc_init = 0.5 * E_max


prices = price_data['VALUE'].resample('H').mean().to_numpy()
prices = prices[:8760]  # Ensure length is 8760
# --------------------------
pv_ac_output_yr1 = solar_output["Year 1"].to_numpy() / 1000000  # Convert W to MW
deg_rate = 0.005  # 0.5% per year
num_years = 30

total_revenue = 0
output = []

for year in range(num_years):
    # Apply PV degradation
    PV = pv_ac_output_yr1 * (1 - deg_rate)**year
    H = len(PV)

    # --------------------------
    # Decision variables
    # --------------------------
    ch_pv = cp.Variable(H)    # battery charging from PV
    ch_grid = cp.Variable(H)  # battery charging from grid
    dis = cp.Variable(H)      # battery discharging
    soc = cp.Variable(H)      # state of charge
    pv_export = cp.Variable(H) # PV exported directly

    # --------------------------
    # Constraints
    # --------------------------
    constraints = []

    # Initial SOC
    constraints.append(soc[0] == soc_init)

    # SOC dynamics
    constraints += [soc[1:] == soc[:-1] + eta_c*(ch_pv[:-1] + ch_grid[:-1]) - dis[:-1]/eta_d]

    # Battery limits
    constraints += [
        soc >= 0,
        soc <= E_max,
        ch_pv >= 0, ch_pv <= P_max,
        ch_grid >= 0, ch_grid <= P_max,
        dis >= 0, dis <= P_max,
        ch_pv + ch_grid + dis <= P_max  # optional: prevent exceeding max power
    ]

    # PV constraints
    constraints += [
        pv_export >= 0,
        pv_export <= PV,
        ch_pv <= PV  # PV charging cannot exceed PV available
    ]

    # --------------------------
    # Objective function
    # --------------------------
    # Revenue = PV export + battery discharge - grid charging cost
    revenue = cp.sum(cp.multiply(prices, pv_export + dis - ch_grid))
    objective = cp.Maximize(revenue)

    # --------------------------
    # Solve optimization
    # --------------------------
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP)

    # --------------------------
    # Record results
    # --------------------------
    revenue_year = revenue.value
    total_revenue += revenue_year
    soc_init = soc.value[-1]  # roll SOC forward

    output.append(revenue_year)
    
    print(f"Year {year+1}: Revenue = ${revenue_year:,.0f}, PV degradation applied")

print(f"\nTotal 30-year revenue = ${total_revenue:,.0f}")


/var/folders/w9/vwctfdqs0v11qkc_klj078540000gn/T/ipykernel_97230/2935266054.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  prices = price_data['VALUE'].resample('H').mean().to_numpy()
/Users/badynmilstein-touesnard/.julia/conda/3/aarch64/lib/python3.12/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


Year 1: Revenue = $425,486, PV degradation applied
Year 2: Revenue = $423,922, PV degradation applied
Year 3: Revenue = $422,450, PV degradation applied
Year 4: Revenue = $420,984, PV degradation applied
Year 5: Revenue = $419,526, PV degradation applied
Year 6: Revenue = $418,075, PV degradation applied
Year 7: Revenue = $416,631, PV degradation applied
Year 8: Revenue = $415,195, PV degradation applied
Year 9: Revenue = $413,765, PV degradation applied
Year 10: Revenue = $412,342, PV degradation applied
Year 11: Revenue = $410,926, PV degradation applied
Year 12: Revenue = $409,517, PV degradation applied
Year 13: Revenue = $408,115, PV degradation applied
Year 14: Revenue = $406,720, PV degradation applied
Year 15: Revenue = $405,331, PV degradation applied
Year 16: Revenue = $403,949, PV degradation applied
Year 17: Revenue = $402,574, PV degradation applied
Year 18: Revenue = $401,205, PV degradation applied
Year 19: Revenue = $399,843, PV degradation applied
Year 20: Revenue = $3

In [76]:
print(len(output))

30


In [77]:
npv = -co_cap  # initial investment
discount_rate = 0.12
om_array = np.array([case_2["Yearly OM"][year] for year in range(1, 31)])


for t in range(1, 29):
    cash_flow = output[t-1] - om_array[t]
    npv += cash_flow / (1 + discount_rate)**t

print(f"30-year NPV = ${npv:,.0f}")


30-year NPV = $-3,055,703


### CASE 3

### CASE 4